In [42]:
import pandas as pd
from pathlib import Path
import os
from datetime import datetime,timedelta
today = datetime.today().date()
yesterday = datetime.now() - timedelta(days=2)


In [43]:
def create_error_file(dataframe: pd.DataFrame, filename: str):
    if dataframe.shape[0]>0:
        output_dir = Path("output")
        output_dir.mkdir(exist_ok=True)
        file_path = output_dir / f"[{today} - Anomaly] {filename}.csv"

        dataframe.to_csv(file_path, index=False)
    
    
    

In [44]:
def extract_compoundname(package,work_phone):
    
    
    if "-" in package:
        compound_name = package.split("-",1)[1]
        if str(compound_name).lower()=='emaar':
            if str(work_phone).lower().startswith('miv'):
                return 'Mivida'
            elif str(work_phone).lower().startswith('utc'):
                return 'Uptown Cairo'
            elif str(work_phone).lower().startswith('mar'):
                return 'Marasi'
        return str(package.split("-",1)[1]).replace(' ','')
    elif "El Gouna" in package:
        return "El Gouna"
    else:
        return None


In [45]:
def extract_provider(package):
    package = str(package).lower()

    bein_keywords = [
        "bein",
        "afcon",
        "euro",
        
    ]

    if any(keyword in package for keyword in bein_keywords):
        return "beIN"

    if "osn" in package:
        return "OSN"
    
    if "fta" in package:
        return "FTA"

    return "N/A"

In [46]:
def read_csv_auto(file_path):
    print(file_path)
    encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1",
        "iso-8859-1",
        "cp1256",   # Arabic Windows
    ]

    for enc in encodings:
        try:
            return pd.read_csv(file_path, dtype=str, encoding=enc)
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Could not read {file_path} with any known encoding.")

In [47]:
def load_files(path,date):

    folder = Path(path)
    files = [f for f in folder.iterdir() if f.is_file()]
    columns = ["SUBSCRIBE_SERVICE_ID",	"SERVICE_MENU_ID",	"SUBSCRIPTION_DATE",	"EXPIRE_DATE",	"WORK_PHONE",	"CUSTOMER_ID",	"NAME",	"DEVICE_ID",	"MAC_ADDRESS",	"SOURCEFILE"]
    all = pd.DataFrame(columns=columns)
    for index, file in enumerate(files, start=1):
        
        
        data =read_csv_auto(os.path.join(folder,file.name))
        
        data['SOURCEFILE'] = file.name
        data['COMPOUND'] = data.apply(lambda row: extract_compoundname(row['NAME'], row['WORK_PHONE']),axis=1)
        compound_from_filename  = (
            file.name
            .replace("CNE_", "")
            .replace("Minerva.csv", "")
            .replace("_", "")
        )

        all = pd.concat([all,data],ignore_index=False)
        all['PROVIDER'] = all['NAME'].apply(extract_provider)
        all['REPORT DATE'] = date
        all = all.loc[all['PROVIDER']=='beIN']
        all.to_csv(f'Orange All {date}.csv')
    return all






Create One File for all days

In [48]:

folder_path = Path(r"C:\Users\mturky\Documents\orange data\19-7-2026")

subfolders = [item for item in folder_path.iterdir() if item.is_dir()]

all_results = []

if subfolders:
    # Process each subfolder
    for folder in subfolders:
        all_results.append(
            load_files(folder, folder.name)
        )
else:
    # Process files directly in the root folder
    all_results.append(
        load_files(folder_path, folder_path.name)
    )

final_df = pd.concat(all_results, ignore_index=True)


final_df['EXPIRE_DATE'] = pd.to_datetime(final_df['EXPIRE_DATE'],dayfirst=True,errors='coerce')
final_df['SUBSCRIPTION_DATE'] = pd.to_datetime(final_df['SUBSCRIPTION_DATE'],dayfirst=True,errors='coerce')


final_df.to_csv(
    r"Orange All 19-7.csv",
    index=False
)

C:\Users\mturky\Documents\orange data\19-7-2026\CNE_EMAARMinerva.csv
C:\Users\mturky\Documents\orange data\19-7-2026\CNE_OrangeMinerva.csv
C:\Users\mturky\Documents\orange data\19-7-2026\CNE_SodicEAST_Minerva.csv
C:\Users\mturky\Documents\orange data\19-7-2026\CNE_SodicETR_Minerva.csv
C:\Users\mturky\Documents\orange data\19-7-2026\CNE_SodicOctoberPlaza_Minerva.csv
C:\Users\mturky\Documents\orange data\19-7-2026\CNE_SodicVillette_Minerva.csv
C:\Users\mturky\Documents\orange data\19-7-2026\CNE_SodicWest_Minerva.csv


C:\Users\mturky\AppData\Local\Temp\1\ipykernel_33176\1417947263.py:22: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  final_df['EXPIRE_DATE'] = pd.to_datetime(final_df['EXPIRE_DATE'],dayfirst=True,errors='coerce')
C:\Users\mturky\AppData\Local\Temp\1\ipykernel_33176\1417947263.py:23: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  final_df['SUBSCRIPTION_DATE'] = pd.to_datetime(final_df['SUBSCRIPTION_DATE'],dayfirst=True,errors='coerce')


In [49]:
final_df.NAME.value_counts()

NAME
beIN Sports-Gouna                                      964
beIN Sports-Emaar                                      754
beIN Sports-NewGiza                                    501
beIN Sports-SodicWest                                  306
beIN Sports-SodicETR                                   187
beIN Sports-SodicVillette                              177
World Cup and 12 months beIN-Emaar                     172
World Cup and 12 months beIN-NewGiza                    99
beIN Sports-OctoberPlaza                                43
beIN Sports-OWEST                                       39
beIN Sports-StonePark                                   26
beIN Sports-MountainView                                18
beIN Sports-FoukaBay                                    15
beIN Sports-Kuwadico                                    13
World Cup and 12 months beIN-SodicVillette              13
beIN Sports-Seashell Playa                              12
beIN Sports-90Avenue                               

In [50]:

final_df = final_df.loc[final_df['SUBSCRIPTION_DATE'].dt.year==2026]

final_df['SUB_MONTH'] =final_df['SUBSCRIPTION_DATE'].dt.to_period('M')

months = sorted(final_df['SUB_MONTH'].dropna().unique())


new_macs_by_month = []

for month in months:
    current_macs = final_df.loc[
            final_df['SUB_MONTH'] == month,
            'MAC_ADDRESS'
        ]
    

    # new_macs = current_macs - seen

    for mac in current_macs:
        new_macs_by_month.append({
            'SUB_MONTH': str(month),
            'MAC_ADDRESS': mac
        })

    

new_macs_df = pd.DataFrame(new_macs_by_month)




In [51]:
new_macs_df.to_csv('new_macs.csv',index=False)


Check for missing MAC ADDRESS

In [52]:
# all_old = load_files(r"C:\Users\mturky\Documents\orange data\1-6-2026", '1-6-2026')
# all_new = load_files(r"C:\Users\mturky\Documents\orange data\8-6-2026",'8-6-2026')

# bein_old = all_old.loc[all_old['NAME'].str.lower().str.contains('bein')]
# bein_new = all_new.loc[all_new['NAME'].str.lower().str.contains('bein')]

# missing = bein_old.loc[~bein_old['MAC_ADDRESS'].isin(bein_new['MAC_ADDRESS'])]
# missing_future_date = missing.loc[missing['EXPIRE_DATE']>'2026-06-01']
# missing_empty_date = missing.loc[missing['EXPIRE_DATE'].isna()]

# create_error_file (missing_future_date,"Missing with future expire date")
# create_error_file(missing_empty_date,"Missing with empty expire date")

In [53]:
# bein_new.columns

Check for missing DUPLICATES

In [54]:
# agg = bein_new.groupby(['WORK_PHONE','MAC_ADDRESS','SOURCEFILE']).agg(count = ('MAC_ADDRESS','count')).reset_index()
# agg = agg.loc[agg['count']>1]
# agg = agg.astype(str)

# create_error_file(agg[['WORK_PHONE','MAC_ADDRESS','SOURCEFILE']], 'Audit sample')

Empty MAC_ADDRESS

In [55]:
# empty_mac = bein_new.loc[bein_new['MAC_ADDRESS'].isna()]
# create_error_file(empty_mac,"Empty MAC ADDRESS")

Empty EXPIRE DATE

In [56]:
# empty_expire = bein_new.loc[bein_new['EXPIRE_DATE'].isna()]
# create_error_file(empty_expire,"Empty EXPIRE DATE")

Expired Files

In [57]:
# empty_expire = bein_new.loc[bein_new['EXPIRE_DATE']< yesterday ]
# create_error_file(empty_expire,"Expired contracts")

In [58]:
# all_new.columns